<a href="https://colab.research.google.com/github/urkmeza/Compressor-Pre-Assembly-Pressure-Validation-Defect-Detection/blob/main/Compressor_anamoly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive.zip


In [2]:
!unzip archive.zip -d compressor_data

Archive:  archive.zip
  inflating: compressor_data/MetroPT3(CompressorDatase).csv  


In [3]:
import os

os.listdir("compressor_data")

['MetroPT3(CompressorDatase).csv']

In [4]:
import pandas as pd
import sqlite3

# SQLite veritabanını oluştur / bağlan
conn = sqlite3.connect("compressor.db")
chunk_size = 200000  # gerekirse 100k'a düşürebiliriz

csv_path = "compressor_data/MetroPT3(CompressorDatase).csv"

i = 0
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    # İlk chunk ise tabloyu yarat, sonrakilerde üzerine ekle
    if_exists_mode = "replace" if i == 0 else "append"
    chunk.to_sql("compressor_raw", conn, if_exists=if_exists_mode, index=False)
    i += 1
    print(f"Chunk {i} yüklendi")

conn.close()

Chunk 1 yüklendi
Chunk 2 yüklendi
Chunk 3 yüklendi
Chunk 4 yüklendi
Chunk 5 yüklendi
Chunk 6 yüklendi
Chunk 7 yüklendi
Chunk 8 yüklendi
Chunk 9 yüklendi
Chunk 10 yüklendi
Chunk 11 yüklendi
Chunk 12 yüklendi
Chunk 13 yüklendi
Chunk 14 yüklendi
Chunk 15 yüklendi
Chunk 16 yüklendi
Chunk 17 yüklendi
Chunk 18 yüklendi
Chunk 19 yüklendi
Chunk 20 yüklendi
Chunk 21 yüklendi
Chunk 22 yüklendi
Chunk 23 yüklendi
Chunk 24 yüklendi
Chunk 25 yüklendi
Chunk 26 yüklendi
Chunk 27 yüklendi
Chunk 28 yüklendi
Chunk 29 yüklendi
Chunk 30 yüklendi
Chunk 31 yüklendi
Chunk 32 yüklendi
Chunk 33 yüklendi
Chunk 34 yüklendi
Chunk 35 yüklendi
Chunk 36 yüklendi
Chunk 37 yüklendi
Chunk 38 yüklendi
Chunk 39 yüklendi
Chunk 40 yüklendi
Chunk 41 yüklendi
Chunk 42 yüklendi
Chunk 43 yüklendi
Chunk 44 yüklendi
Chunk 45 yüklendi
Chunk 46 yüklendi
Chunk 47 yüklendi
Chunk 48 yüklendi
Chunk 49 yüklendi
Chunk 50 yüklendi
Chunk 51 yüklendi
Chunk 52 yüklendi
Chunk 53 yüklendi
Chunk 54 yüklendi
Chunk 55 yüklendi
Chunk 56 yüklendi
C

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("compressor.db")

df_sample = pd.read_sql_query("""
    SELECT *
    FROM compressor_raw
    LIMIT 100000;
""", conn)

df_sample.shape
df_sample.head()

,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,1,2020-02-01 00:00:01,-0.012,9.358,9.342,-0.022,9.360,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,2,2020-02-01 00:00:02,-0.012,9.356,9.340,-0.022,9.358,53.625,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,3,2020-02-01 00:00:03,-0.012,9.356,9.340,-0.022,9.358,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,4,2020-02-01 00:00:04,-0.012,9.354,9.338,-0.022,9.356,53.625,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


In [6]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("compressor.db")

info_df = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM compressor_raw;",
    conn
)
cols_df = pd.read_sql_query(
    "PRAGMA table_info(compressor_raw);",
    conn
)

info_df, cols_df

(   row_count
 0   15169480,
     cid             name     type  notnull dflt_value  pk
 0     0       Unnamed: 0  INTEGER        0       None   0
 1     1        timestamp     TEXT        0       None   0
 2     2              TP2     REAL        0       None   0
 3     3              TP3     REAL        0       None   0
 4     4               H1     REAL        0       None   0
 5     5      DV_pressure     REAL        0       None   0
 6     6       Reservoirs     REAL        0       None   0
 7     7  Oil_temperature     REAL        0       None   0
 8     8    Motor_current     REAL        0       None   0
 9     9             COMP     REAL        0       None   0
 10   10       DV_eletric     REAL        0       None   0
 11   11           Towers     REAL        0       None   0
 12   12              MPG     REAL        0       None   0
 13   13              LPS     REAL        0       None   0
 14   14  Pressure_switch     REAL        0       None   0
 15   15        Oil_level  

In [7]:
stats_df = pd.read_sql_query("""
SELECT
  COUNT(*) AS n_rows,
  AVG(TP2) AS avg_tp2,
  MIN(TP2) AS min_tp2,
  MAX(TP2) AS max_tp2,
  AVG(TP3) AS avg_tp3,
  MIN(TP3) AS min_tp3,
  MAX(TP3) AS max_tp3,
  AVG(Motor_current) AS avg_motor_current,
  MIN(Motor_current) AS min_motor_current,
  MAX(Motor_current) AS max_motor_current
FROM compressor_raw;
""", conn)

stats_df

,n_rows,avg_tp2,min_tp2,max_tp2,avg_tp3,min_tp3,max_tp3,avg_motor_current,min_motor_current,max_motor_current
0,15169480,1.367726,-0.034,10.676,8.984597,0.69,10.31,2.050273,0.02,9.8075


In [8]:
tp2_anoms = pd.read_sql_query("""
WITH tp2_stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    STDDEV(TP2) AS std_tp2
  FROM compressor_raw
),
tp2_z AS (
  SELECT
    timestamp,
    TP2,
    (TP2 - (SELECT mean_tp2 FROM tp2_stats))
      / NULLIF((SELECT std_tp2 FROM tp2_stats), 0) AS z_tp2
  FROM compressor_raw
)
SELECT *
FROM tp2_z
WHERE ABS(z_tp2) > 3   -- 3 std sapma üstü anomaliler
ORDER BY ABS(z_tp2) DESC
LIMIT 100;
""", conn)

tp2_anoms.head()

DatabaseError: Execution failed on sql '
WITH tp2_stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    STDDEV(TP2) AS std_tp2
  FROM compressor_raw
),
tp2_z AS (
  SELECT
    timestamp,
    TP2,
    (TP2 - (SELECT mean_tp2 FROM tp2_stats))
      / NULLIF((SELECT std_tp2 FROM tp2_stats), 0) AS z_tp2
  FROM compressor_raw
)
SELECT *
FROM tp2_z
WHERE ABS(z_tp2) > 3   -- 3 std sapma üstü anomaliler
ORDER BY ABS(z_tp2) DESC
LIMIT 100;
': no such function: STDDEV

In [9]:
tp2_anoms = pd.read_sql_query("""
WITH tp2_stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    STDDEV(TP2) AS std_tp2
  FROM compressor_raw
),
tp2_z AS (
  SELECT
    timestamp,
    TP2,
    (TP2 - (SELECT mean_tp2 FROM tp2_stats))
      / NULLIF((SELECT std_tp2 FROM tp2_stats), 0) AS z_tp2
  FROM compressor_raw
)
SELECT *
FROM tp2_z
WHERE ABS(z_tp2) > 3
ORDER BY ABS(z_tp2) DESC
LIMIT 100;
""", conn)

tp2_anoms.head()

DatabaseError: Execution failed on sql '
WITH tp2_stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    STDDEV(TP2) AS std_tp2
  FROM compressor_raw
),
tp2_z AS (
  SELECT
    timestamp,
    TP2,
    (TP2 - (SELECT mean_tp2 FROM tp2_stats))
      / NULLIF((SELECT std_tp2 FROM tp2_stats), 0) AS z_tp2
  FROM compressor_raw
)
SELECT *
FROM tp2_z
WHERE ABS(z_tp2) > 3
ORDER BY ABS(z_tp2) DESC
LIMIT 100;
': no such function: STDDEV

In [10]:
tp2_anoms = pd.read_sql_query("""
WITH tp2_stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    -- stddev hesapla: sqrt(E[x^2] - (E[x])^2)
    sqrt(AVG(TP2 * TP2) - AVG(TP2) * AVG(TP2)) AS std_tp2
  FROM compressor_raw
),
tp2_z AS (
  SELECT
    timestamp,
    TP2,
    (TP2 - (SELECT mean_tp2 FROM tp2_stats))
      / NULLIF((SELECT std_tp2 FROM tp2_stats), 0) AS z_tp2
  FROM compressor_raw
)
SELECT *
FROM tp2_z
WHERE ABS(z_tp2) > 3
ORDER BY ABS(z_tp2) DESC
LIMIT 100;
""", conn)

tp2_anoms.head()

,timestamp,TP2,z_tp2


In [11]:
tp3_anoms = pd.read_sql_query("""
WITH tp3_stats AS (
  SELECT
    AVG(TP3) AS mean_tp3,
    sqrt(AVG(TP3 * TP3) - AVG(TP3) * AVG(TP3)) AS std_tp3
  FROM compressor_raw
),
tp3_z AS (
  SELECT
    timestamp,
    TP3,
    (TP3 - (SELECT mean_tp3 FROM tp3_stats))
      / NULLIF((SELECT std_tp3 FROM tp3_stats), 0) AS z_tp3
  FROM compressor_raw
)
SELECT *
FROM tp3_z
WHERE ABS(z_tp3) > 3
ORDER BY ABS(z_tp3) DESC
LIMIT 100;
""", conn)

tp3_anoms.head()

,timestamp,TP3,z_tp3
0,2020-03-28 23:05:27,0.690,-12.976998
1,2020-03-28 23:05:28,0.702,-12.958224
2,2020-03-28 23:05:29,0.710,-12.945708
3,2020-03-28 23:05:30,0.718,-12.933192
4,2020-03-28 23:05:31,0.724,-12.923804


In [12]:
both_anoms = pd.read_sql_query("""
WITH stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    sqrt(AVG(TP2 * TP2) - AVG(TP2) * AVG(TP2)) AS std_tp2,
    AVG(TP3) AS mean_tp3,
    sqrt(AVG(TP3 * TP3) - AVG(TP3) * AVG(TP3)) AS std_tp3
  FROM compressor_raw
),
z_scores AS (
  SELECT
    timestamp,
    TP2,
    TP3,
    (TP2 - mean_tp2) / NULLIF(std_tp2, 0) AS z_tp2,
    (TP3 - mean_tp3) / NULLIF(std_tp3, 0) AS z_tp3
  FROM compressor_raw, stats
)
SELECT
  *,
  CASE
    WHEN ABS(z_tp2) > 3 AND ABS(z_tp3) > 3 THEN 1
    ELSE 0
  END AS is_anomaly
FROM z_scores
WHERE ABS(z_tp2) > 3 OR ABS(z_tp3) > 3
LIMIT 1000;
""", conn)

both_anoms.head()

,timestamp,TP2,TP3,z_tp2,z_tp3,is_anomaly
0,2020-03-03 02:51:36,4.394,3.796,0.930895,-8.117623,0
1,2020-03-03 02:51:37,4.400,3.804,0.932740,-8.105107,0
2,2020-03-03 02:51:38,4.408,3.812,0.935201,-8.092591,0
3,2020-03-03 02:51:39,4.422,3.830,0.939508,-8.064430,0
4,2020-03-03 02:51:40,4.428,3.838,0.941353,-8.051913,0


In [13]:
pd.read_sql_query("""
CREATE TABLE IF NOT EXISTS compressor_labeled AS
WITH stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    sqrt(AVG(TP2 * TP2) - AVG(TP2) * AVG(TP2)) AS std_tp2,
    AVG(TP3) AS mean_tp3,
    sqrt(AVG(TP3 * TP3) - AVG(TP3) * AVG(TP3)) AS std_tp3
  FROM compressor_raw
),
z_scores AS (
  SELECT
    timestamp,
    TP2,
    TP3,
    Reservoirs,
    Oil_temperature,
    Motor_current,
    (TP2 - mean_tp2) / NULLIF(std_tp2, 0) AS z_tp2,
    (TP3 - mean_tp3) / NULLIF(std_tp3, 0) AS z_tp3
  FROM compressor_raw, stats
)
SELECT
  *,
  CASE
    WHEN ABS(z_tp2) > 3 AND ABS(z_tp3) > 3 THEN 1
    ELSE 0
  END AS is_anomaly
FROM z_scores;
""", conn)

TypeError: 'NoneType' object is not iterable

In [14]:
sample_labeled = pd.read_sql_query("""
SELECT *
FROM compressor_labeled
LIMIT 1000;
""", conn)

sample_labeled['is_anomaly'].value_counts()

,count
is_anomaly,
0,1000


In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("compressor.db")

df = pd.read_sql_query("""
SELECT
  timestamp,
  TP2,
  TP3,
  Reservoirs,
  Oil_temperature,
  Motor_current,
  is_anomaly
FROM compressor_labeled
LIMIT 100000;
""", conn)

df.head(), df.shape, df["is_anomaly"].value_counts()

(             timestamp    TP2    TP3  Reservoirs  Oil_temperature  \
 0  2020-02-01 00:00:00 -0.012  9.358       9.358           53.600   
 1  2020-02-01 00:00:01 -0.012  9.358       9.360           53.600   
 2  2020-02-01 00:00:02 -0.012  9.356       9.358           53.625   
 3  2020-02-01 00:00:03 -0.012  9.356       9.358           53.600   
 4  2020-02-01 00:00:04 -0.012  9.354       9.356           53.625   
 
    Motor_current  is_anomaly  
 0         0.0400           0  
 1         0.0400           0  
 2         0.0400           0  
 3         0.0425           0  
 4         0.0400           0  ,
 (100000, 7),
 is_anomaly
 0    100000
 Name: count, dtype: int64)

In [16]:
df.describe()
df.isna().sum()

,0
timestamp,0
TP2,0
TP3,0
Reservoirs,0
Oil_temperature,0
Motor_current,0
is_anomaly,0


In [17]:
df["is_anomaly"].value_counts(normalize=True)

,proportion
is_anomaly,
0,1.0


In [18]:
pd.read_sql_query("""
DROP TABLE IF EXISTS compressor_labeled;
""", conn)

pd.read_sql_query("""
CREATE TABLE compressor_labeled AS
WITH stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    sqrt(AVG(TP2 * TP2) - AVG(TP2) * AVG(TP2)) AS std_tp2,
    AVG(TP3) AS mean_tp3,
    sqrt(AVG(TP3 * TP3) - AVG(TP3) * AVG(TP3)) AS std_tp3
  FROM compressor_raw
),
z_scores AS (
  SELECT
    timestamp,
    TP2,
    TP3,
    Reservoirs,
    Oil_temperature,
    Motor_current,
    (TP2 - mean_tp2) / NULLIF(std_tp2, 0) AS z_tp2,
    (TP3 - mean_tp3) / NULLIF(std_tp3, 0) AS z_tp3
  FROM compressor_raw, stats
)
SELECT
  *,
  CASE
    WHEN ABS(z_tp2) > 2.5 OR ABS(z_tp3) > 2.5 THEN 1
    ELSE 0
  END AS is_anomaly
FROM z_scores;
""", conn)

TypeError: 'NoneType' object is not iterable

In [19]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("compressor.db")
cursor = conn.cursor()

# 1) Eski tabloyu sil
cursor.execute("DROP TABLE IF EXISTS compressor_labeled;")
conn.commit()

# 2) Yeni eşiğe göre tabloyu oluştur
cursor.execute("""
CREATE TABLE compressor_labeled AS
WITH stats AS (
  SELECT
    AVG(TP2) AS mean_tp2,
    sqrt(AVG(TP2 * TP2) - AVG(TP2) * AVG(TP2)) AS std_tp2,
    AVG(TP3) AS mean_tp3,
    sqrt(AVG(TP3 * TP3) - AVG(TP3) * AVG(TP3)) AS std_tp3
  FROM compressor_raw
),
z_scores AS (
  SELECT
    timestamp,
    TP2,
    TP3,
    Reservoirs,
    Oil_temperature,
    Motor_current,
    (TP2 - mean_tp2) / NULLIF(std_tp2, 0) AS z_tp2,
    (TP3 - mean_tp3) / NULLIF(std_tp3, 0) AS z_tp3
  FROM compressor_raw, stats
)
SELECT
  *,
  CASE
    WHEN ABS(z_tp2) > 2.5 OR ABS(z_tp3) > 2.5 THEN 1
    ELSE 0
  END AS is_anomaly
FROM z_scores;
""")
conn.commit()

# 3) Kontrol: yeni dağılım
df = pd.read_sql_query("""
SELECT
  TP2, TP3, Reservoirs, Oil_temperature, Motor_current, is_anomaly
FROM compressor_labeled
LIMIT 100000;
""", conn)

df["is_anomaly"].value_counts(normalize=True)

,proportion
is_anomaly,
0,0.97119
1,0.02881


In [20]:
from sklearn.model_selection import train_test_split

X = df[["TP2", "TP3", "Reservoirs", "Oil_temperature", "Motor_current"]]
y = df["is_anomaly"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

(is_anomaly
 0    0.971187
 1    0.028813
 Name: proportion, dtype: float64,
 is_anomaly
 0    0.9712
 1    0.0288
 Name: proportion, dtype: float64)

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",   # dengesiz sınıflar için
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19424
           1       1.00      1.00      1.00       576

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

[[19424     0]
 [    0   576]]


In [22]:
import pandas as pd
import numpy as np

importances = model.feature_importances_
feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp

,feature,importance
0,TP2,0.528164
4,Motor_current,0.349634
1,TP3,0.074714
2,Reservoirs,0.044737
3,Oil_temperature,0.002751
